# Solana Transfer Data Overview

Academy City Bank — Algorithm Risk Control Analysis  
Data: ~88 ranges ≈ 2 days (slots 414455000–414894999)

**Sections**
1. Environment setup
2. Data quality checks
3. SOL transfer overview
4. Token transfer overview
5. Address-level feature matrix
6. KMeans clustering
7. t-SNE / UMAP visualization
8. Preliminary risk signals

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils.ch_client import query_df
sns.set_theme(style='darkgrid', palette='muted')
pd.set_option('display.float_format', '{:,.2f}'.format)
print('libs OK')

libs OK


## 1. Data Quality Checks

In [ ]:
q_quality = """
SELECT 'sol' AS tbl, count() AS rows,
    countIf(from_address='') AS from_empty, countIf(to_address='') AS to_empty,
    min(slot) AS slot_min, max(slot) AS slot_max, countDistinct(slot) AS unique_slots
FROM default.test_sol_transfer_local
UNION ALL
SELECT 'token' AS tbl, count() AS rows,
    countIf(from_owner='') AS from_empty, countIf(to_owner='') AS to_empty,
    min(slot) AS slot_min, max(slot) AS slot_max, countDistinct(slot) AS unique_slots
FROM default.test_token_transfer_local
"""
display(query_df(q_quality))

In [ ]:
q_owner = """
SELECT
    countIf(from_owner!='') / count() AS from_owner_fill_rate,
    countIf(to_owner!='')   / count() AS to_owner_fill_rate
FROM default.test_token_transfer_local
"""
display(query_df(q_owner))

## 2. SOL Transfer Overview

In [ ]:
q_sol_hourly = """
SELECT toStartOfHour(timestamp) AS hour,
       count() AS transfers, sum(lamports) AS total_lamports
FROM default.test_sol_transfer_local
GROUP BY hour ORDER BY hour
"""
sol_h = query_df(q_sol_hourly)
sol_h['total_sol'] = sol_h['total_lamports'].astype(float) / 1e9
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].bar(sol_h['hour'], sol_h['transfers'].astype(float), width=0.04, color='steelblue')
axes[0].set(title='SOL Transfers per Hour', ylabel='Count')
axes[1].bar(sol_h['hour'], sol_h['total_sol'], width=0.04, color='orange')
axes[1].set(title='Total SOL Volume per Hour', ylabel='SOL')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
q_sol_dist = "SELECT lamports FROM default.test_sol_transfer_local WHERE lamports>0 ORDER BY rand() LIMIT 500000"
sol_s = query_df(q_sol_dist)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log10(sol_s['lamports'].astype(float)+1), bins=60, color='steelblue', edgecolor='white')
ax.set(xlabel='log10(lamports)', ylabel='Count', title='SOL Lamport Distribution (500K sample)')
plt.tight_layout(); plt.show()

## 3. Token Transfer Overview

In [ ]:
q_tok_hourly = """
SELECT toStartOfHour(timestamp) AS hour, count() AS transfers, program_id
FROM default.test_token_transfer_local
GROUP BY hour, program_id ORDER BY hour
"""
tok_h = query_df(q_tok_hourly)
pivoted = tok_h.pivot_table(index='hour', columns='program_id', values='transfers', fill_value=0)
pivoted.plot.bar(stacked=True, figsize=(14, 5), color=['steelblue','tomato'])
plt.title('Token Transfers per Hour by Program'); plt.ylabel('Count')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
q_top_mint = """
SELECT mint, count() AS cnt FROM default.test_token_transfer_local
WHERE mint!='' GROUP BY mint ORDER BY cnt DESC LIMIT 30
"""
top_mints = query_df(q_top_mint)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_mints['mint'], top_mints['cnt'].astype(float), color='mediumseagreen')
ax.set(xlabel='Transfer Count', title='Top 30 Mints by Transfer Count')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
q_type = "SELECT transfer_type, count() AS cnt FROM default.test_token_transfer_local GROUP BY transfer_type"
tok_types = query_df(q_type)
tok_types.set_index('transfer_type')['cnt'].astype(float).plot.pie(autopct='%.1f%%', figsize=(6,6))
plt.title('Token Transfer Type Distribution'); plt.ylabel('')
plt.tight_layout(); plt.show()

## 4. Address-Level Feature Matrix

In [ ]:
q_sol_feat = """
SELECT wallet,
       out_count, in_count, out_count+in_count AS total_activity,
       total_out_lamports/1e9 AS total_out_sol, total_in_lamports/1e9 AS total_in_sol,
       unique_out_cp, unique_in_cp, avg_out_lamports/1e9 AS avg_out_sol
FROM (
    SELECT from_address AS wallet, count() AS out_count,
           sum(lamports) AS total_out_lamports, avg(lamports) AS avg_out_lamports,
           uniq(to_address) AS unique_out_cp
    FROM default.test_sol_transfer_local GROUP BY from_address
) o LEFT JOIN (
    SELECT to_address AS wallet, count() AS in_count,
           sum(lamports) AS total_in_lamports, uniq(from_address) AS unique_in_cp
    FROM default.test_sol_transfer_local GROUP BY to_address
) i USING wallet
ORDER BY total_activity DESC LIMIT 50000
"""
sol_feat = query_df(q_sol_feat).fillna(0)
print(f'SOL feature matrix: {sol_feat.shape}')
display(sol_feat.describe())

In [ ]:
q_tok_feat = """
SELECT wallet,
       out_count+in_count AS total_activity, out_count, in_count,
       unique_mints_out+unique_mints_in AS unique_mints,
       unique_out_cp, unique_in_cp
FROM (
    SELECT from_owner AS wallet, count() AS out_count,
           uniq(mint) AS unique_mints_out, uniq(to_owner) AS unique_out_cp
    FROM default.test_token_transfer_local WHERE from_owner!=''
    GROUP BY from_owner
) o LEFT JOIN (
    SELECT to_owner AS wallet, count() AS in_count,
           uniq(mint) AS unique_mints_in, uniq(from_owner) AS unique_in_cp
    FROM default.test_token_transfer_local WHERE to_owner!=''
    GROUP BY to_owner
) i USING wallet
ORDER BY total_activity DESC LIMIT 50000
"""
tok_feat = query_df(q_tok_feat).fillna(0)
print(f'Token feature matrix: {tok_feat.shape}')
display(tok_feat.describe())

## 5. KMeans Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SOL_COLS = ['out_count','in_count','total_out_sol','total_in_sol','unique_out_cp','unique_in_cp','avg_out_sol']
X_sol = sol_feat[SOL_COLS].values.astype(float)
X_sol_s = StandardScaler().fit_transform(np.log1p(np.clip(X_sol, 0, None)))

inertias, silhouettes = [], []
K_RANGE = range(3, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_sol_s)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sol_s, lbl, sample_size=5000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), inertias, marker='o'); axes[0].set(title='Elbow (SOL)', xlabel='k', ylabel='Inertia')
axes[1].plot(list(K_RANGE), silhouettes, marker='o', color='tomato'); axes[1].set(title='Silhouette (SOL)', xlabel='k')
plt.tight_layout(); plt.show()

In [ ]:
BEST_K_SOL = 5  # adjust after inspecting elbow/silhouette
km_sol = KMeans(n_clusters=BEST_K_SOL, random_state=42, n_init=10)
sol_feat['cluster'] = km_sol.fit_predict(X_sol_s)
print('SOL cluster sizes:\n', sol_feat['cluster'].value_counts().sort_index())
display(sol_feat.groupby('cluster')[SOL_COLS].median())

In [ ]:
TOK_COLS = ['out_count','in_count','unique_mints','unique_out_cp','unique_in_cp']
X_tok = tok_feat[TOK_COLS].values.astype(float)
X_tok_s = StandardScaler().fit_transform(np.log1p(np.clip(X_tok, 0, None)))

BEST_K_TOK = 5
km_tok = KMeans(n_clusters=BEST_K_TOK, random_state=42, n_init=10)
tok_feat['cluster'] = km_tok.fit_predict(X_tok_s)
print('Token cluster sizes:\n', tok_feat['cluster'].value_counts().sort_index())
display(tok_feat.groupby('cluster')[TOK_COLS].median())

## 6. 2-D Visualization (t-SNE / UMAP)

In [ ]:
from sklearn.manifold import TSNE

N = 8000
rng = np.random.default_rng(42)
idx = rng.choice(len(X_sol_s), size=min(N, len(X_sol_s)), replace=False)
X_sub = X_sol_s[idx]
lbl_sub = sol_feat['cluster'].values[idx]

embed = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_sub)

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(embed[:,0], embed[:,1], c=lbl_sub, cmap='tab10', alpha=0.5, s=5)
plt.colorbar(sc, ax=ax, label='Cluster')
ax.set(title=f't-SNE SOL Address Features (k={BEST_K_SOL})', xlabel='dim-1', ylabel='dim-2')
plt.tight_layout(); plt.show()

In [ ]:
try:
    import umap
    embed_umap = umap.UMAP(n_components=2, random_state=42).fit_transform(X_sub)
    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(embed_umap[:,0], embed_umap[:,1], c=lbl_sub, cmap='tab10', alpha=0.5, s=5)
    plt.colorbar(sc, ax=ax, label='Cluster')
    ax.set(title=f'UMAP SOL Address Features (k={BEST_K_SOL})')
    plt.tight_layout(); plt.show()
except ImportError:
    print('umap-learn not installed; skipping. Run: pip install umap-learn')

## 7. Preliminary Risk Signals

In [ ]:
# High-frequency senders: top per-hour tx counts
q_hf = """
SELECT from_address AS wallet,
       toStartOfHour(timestamp) AS hour,
       count() AS tx_per_hour
FROM default.test_sol_transfer_local
GROUP BY wallet, hour
ORDER BY tx_per_hour DESC LIMIT 50
"""
display(query_df(q_hf).head(20))

In [ ]:
# Large SOL transfers (> 99th percentile)
q_large = """
SELECT slot, signature, from_address, to_address, lamports/1e9 AS sol_amount
FROM default.test_sol_transfer_local
WHERE lamports > (SELECT quantileExact(0.99)(lamports) FROM default.test_sol_transfer_local)
ORDER BY lamports DESC LIMIT 50
"""
display(query_df(q_large).head(20))

In [ ]:
# Wallets with >=5 transfers in a single slot (potential bot)
q_multi = """
SELECT from_address AS wallet, slot, count() AS transfers_in_slot
FROM default.test_sol_transfer_local
GROUP BY wallet, slot
HAVING transfers_in_slot >= 5
ORDER BY transfers_in_slot DESC LIMIT 50
"""
display(query_df(q_multi).head(20))
print('Analysis complete.')